In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
modify_path = "../CineIq_Data/modify/"
cineiq_df = pd.read_csv(modify_path + "cineiq_metadata.csv").fillna("")

In [3]:
cineiq_df.head()

,movieId,title,overview,genres,keywords
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,jealousy toy boy friendship friends rivalry bo...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,board game disappearance based on children's b...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Romance Comedy,fishing best friend duringcreditsstinger old men
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,based on novel interracial relationship single...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...


In [4]:
with open(modify_path + "svd_model.pkl","rb") as f:
    svd_model=pickle.load(f)

In [5]:
cineiq_df["soup"] = (
    cineiq_df["overview"] + " " +
    (cineiq_df["genres"] + " ") * 2 +
    (cineiq_df["keywords"] + " ") * 3
)
cineiq_df = cineiq_df.fillna("")

In [6]:
tfidf = TfidfVectorizer(stop_words="english", min_df=4,max_features=20000)
tfidf_matrix = tfidf.fit_transform(cineiq_df["soup"])

In [7]:
cineiq_df.head()

,movieId,title,overview,genres,keywords,soup
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,jealousy toy boy friendship friends rivalry bo...,"Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,board game disappearance based on children's b...,When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Romance Comedy,fishing best friend duringcreditsstinger old men,A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,based on novel interracial relationship single...,"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Just when George Banks has recovered from his ...


In [8]:
def search_movie(query, df):
    matches = df[df["title"].str.lower().str.contains(query.lower(),na=False)]
    if matches.empty:
        return "No movies found."
    return matches[["movieId","title"]].head(10)

In [9]:
def get_hybrid_recommendations(user_id, target_idx, df, tfidf_matrix, svd_model, n=10, svd_weight = 0.7, content_weight = 0.3):
    target_movie_id = df.iloc[target_idx]["movieId"]
    matched_title = df.iloc[target_idx]["title"]

    movie_vector = tfidf_matrix[target_idx]
    sim_scores = cosine_similarity(movie_vector, tfidf_matrix).flatten()

    candidate_indices = sim_scores.argsort()[::-1]
    candidate_indices = candidate_indices[1:51]

    candidates = df.iloc[candidate_indices][["movieId","title"]].copy()
    candidates["content_score"] = sim_scores[candidate_indices]

    svd_preds = []
    for m_id in candidates["movieId"]:
        raw_pred = svd_model.predict(uid=user_id, iid=m_id).est
        normalised_svd = (raw_pred - 0.5)/4.5
        svd_preds.append(normalised_svd)

    candidates["svd_score"] = svd_preds

    candidates["hybrid_score"] = (candidates["svd_score"] * svd_weight) + (candidates["content_score"] * content_weight)

    final_recs = candidates.sort_values(by="hybrid_score",ascending=False).head(n)

    return final_recs[["movieId","title","content_score","svd_score","hybrid_score"]].reset_index(drop=True)

In [10]:
search_movie("spider-man",cineiq_df)

,movieId,title
5228,5349,Spider-Man (2002)
7916,8636,Spider-Man 2 (2004)
11557,52722,Spider-Man 3 (2007)
14517,76709,Spider-Man: The Ultimate Villain Showdown (2002)
18167,95510,"Amazing Spider-Man, The (2012)"
21253,110553,The Amazing Spider-Man 2 (2014)


In [11]:
get_hybrid_recommendations(1,11557,cineiq_df,tfidf_matrix,svd_model)

,movieId,title,content_score,svd_score,hybrid_score
0,8636,Spider-Man 2 (2004),0.437458,0.753090,0.658400
1,89745,"Avengers, The (2012)",0.270141,0.785725,0.631050
2,110102,Captain America: The Winter Soldier (2014),0.191768,0.801627,0.618669
3,168252,Logan (2017),0.185875,0.792536,0.610538
4,8961,"Incredibles, The (2004)",0.217385,0.777918,0.609758
5,122904,Deadpool (2016),0.197459,0.785084,0.608796
6,122916,Thor: Ragnarok (2017),0.258184,0.756477,0.606989
7,5349,Spider-Man (2002),0.381375,0.700979,0.605098
8,122922,Doctor Strange (2016),0.239299,0.745919,0.593933
9,122892,Avengers: Age of Ultron (2015),0.300212,0.711657,0.588223
